<a href="https://colab.research.google.com/github/caroarcrz/Bin-packing-problem-Contenedor-Refrigerado/blob/main/Contenedor_refigerado1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizando el llenado de un contenedor refrigerado

El problema de la optimización en el llenado de un contenedor refrigerado consiste en determinar el acomodo de un
conjunto de cajas de frutas y verduras dentro de un contenedor utilizado para transporte terrestre. El contenedor cuenta
con una unidad de enfriamiento ubicada en la parte frontal, cercana a la cabina del conductor, por lo que la temperatura
no se distribuye de manera uniforme. En general, las zonas cercanas al sistema de refrigeración presentan temperaturas
más bajas, mientras que las zonas cercanas a la puerta trasera tienden a presentar temperaturas más elevadas

Para representar este comportamiento térmico de forma sencilla, el contenedor se divide en un conjunto de zonas
térmicas. Cada zona térmica contiene un número determinado de hileras, cada hilera permite acomodar un número
máximo de cajas a lo ancho, y cada posición puede tener varios niveles de apilamiento. De esta manera, el contenedor
se representa como un conjunto de posiciones discretas donde pueden colocarse cajas.

Cada tipo de producto tiene una cantidad fija de cajas que debe ser transportada, así como características propias
de peso, dimensiones, resistencia al apilamiento y sensibilidad térmica. Los productos más sensibles a la temperatura
deben colocarse preferentemente en las zonas más frías del contenedor, mientras que los productos menos sensibles
pueden ubicarse en zonas de mayor temperatura. El objetivo del problema es asignar las cajas de cada producto a las
posiciones disponibles dentro del contenedor, minimizando el deterioro térmico total y respetando las restricciones
físicas de capacidad, apilamiento y factibilidad de carga.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#PRIMERO SE INSTALARAN LAS PAQUETERIAS QUE SE UTILIZARAN PARA EL MODELADO

In [ ]:
!pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 46.7 MB/s eta 0:00:00


In [ ]:
!pip install plotly

In [ ]:
####LIBRERIAS

import gurobipy as gp
from gurobipy import GRB

import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px


# Conjuntos

$P$: conjunto de tipos de productos.

$Z$: conjunto de zonas térmicas del contenedor.

$H_z$: conjunto de hileras disponibles en la zona térmica $z\in Z$.

$A_z$: conjunto de posiciones a lo ancho disponibles en cada hilera de la zona $z\in Z$.

$L_z$: conjunto de niveles de apilamiento disponibles en la zona $z\in Z$.


# Parámetros

$n_p$: número de cajas del producto $p\in P$ que deben transportarse.

$w_p$: peso de una caja del producto $p\in P$.

$s_p$: peso máximo que puede soportar una caja del producto $p\in P$ encima de ella.

$\alpha_p$: coeficiente de deterioro térmico del producto $p\in P$.

$\theta_z$: nivel de temperatura asociado a la zona térmica $z\in Z$.

$d_{pz}$: penalización por colocar una caja del producto $p\in P$ en la zona térmica $z\in Z$.

# Archivo de Excel
Aquí se agrega un código para subir las instancias en un archivo de formato .xlsx que contiene los parámetros y conjuntos a utilizar.

In [ ]:
####Esta parte del código es para subir el código y saber el nombre
### de las hojas que se usan en excel.

from google.colab import files
import pandas as pd

uploaded = files.upload()

archivo = next(iter(uploaded))

xls = pd.ExcelFile(archivo)
sheet_names = xls.sheet_names

print(f"El nombre de las hojas son '{archivo}': {sheet_names}")



Saving instancias4.xlsx to instancias4 (1).xlsx
El nombre de las hojas son 'instancias4 (1).xlsx': ['Contenedor', 'Productos', 'Zonas']


In [ ]:
from google.colab import files
#En esta parte del código se sube el archivo .xlsx
#que contenga las instancias que se utilizarán.

uploaded = files.upload()

archivo = next(iter(uploaded))

contenedor = pd.read_excel(archivo, sheet_name="Contenedor")
productos = pd.read_excel(archivo, sheet_name="Productos")
zonas = pd.read_excel(archivo, sheet_name="Zonas")

####Limpieza de columnas en el excel
zonas.columns = zonas.columns.str.strip()

param = dict(zip(contenedor.Conjunto, contenedor.Valor))

Z = range(1, param["Z"]+1)
H = range(1, param["H"]+1)
A = range(1, param["A"]+1)
L = range(1, param["L"]+1)


P = productos["Producto"].tolist()

n = dict(zip(productos["Producto"], productos.n))
w = dict(zip(productos["Producto"], productos.w))
s = dict(zip(productos["Producto"], productos.s))
alpha = dict(zip(productos["Producto"], productos.alpha))

theta = dict(zip(zonas['Zona'], zonas['theta']))

Saving instancias4.xlsx to instancias4 (2).xlsx


In [ ]:
#Imprimimos la lista de datos presentada en el archivo .xlsx
print("L =", list(L))
print("Z =", list(Z))
print("H =", list(H))
print("A =", list(A))
print("P =", P)
print("n =", n)
print("w =", w)
print("s =", s)
print("alpha =", alpha)

L = [1, 2, 3, 4]
Z = [1, 2, 3, 4]
H = [1]
A = [1, 2, 3]
P = ['Fresa (F)', 'Papa (P)', 'Jitomate (J)', 'Manzana (M)', 'Lechuga (L)', 'Cebolla (C)', 'Zanahoria (ZN)', 'Pepino (PE)']
n = {'Fresa (F)': 8, 'Papa (P)': 10, 'Jitomate (J)': 6, 'Manzana (M)': 3, 'Lechuga (L)': 4, 'Cebolla (C)': 6, 'Zanahoria (ZN)': 5, 'Pepino (PE)': 5}
w = {'Fresa (F)': 4, 'Papa (P)': 7, 'Jitomate (J)': 5, 'Manzana (M)': 6, 'Lechuga (L)': 3, 'Cebolla (C)': 5, 'Zanahoria (ZN)': 4, 'Pepino (PE)': 4}
s = {'Fresa (F)': 10, 'Papa (P)': 20, 'Jitomate (J)': 15, 'Manzana (M)': 18, 'Lechuga (L)': 8, 'Cebolla (C)': 16, 'Zanahoria (ZN)': 12, 'Pepino (PE)': 11}
alpha = {'Fresa (F)': 10, 'Papa (P)': 2, 'Jitomate (J)': 8, 'Manzana (M)': 7, 'Lechuga (L)': 12, 'Cebolla (C)': 3, 'Zanahoria (ZN)': 5, 'Pepino (PE)': 9}


# PENALIZACION
$d_{p,z}=\alpha_{p}*θ_z$


In [ ]:

#Penalización
d = {} ####Se deja el conjunto vacío para que vaya guardando resultados

for p in P:   #Se comienza la iteración de los tres productos
    for z in Z: # Son 3 zonas las que va a recorrer
        d[p,z] = alpha[p]*theta[z]


#Imprimimos el proceso de la iteración
print(d)


{('Fresa (F)', 1): 20, ('Fresa (F)', 2): 50, ('Fresa (F)', 3): 80, ('Fresa (F)', 4): 100, ('Papa (P)', 1): 4, ('Papa (P)', 2): 10, ('Papa (P)', 3): 16, ('Papa (P)', 4): 20, ('Jitomate (J)', 1): 16, ('Jitomate (J)', 2): 40, ('Jitomate (J)', 3): 64, ('Jitomate (J)', 4): 80, ('Manzana (M)', 1): 14, ('Manzana (M)', 2): 35, ('Manzana (M)', 3): 56, ('Manzana (M)', 4): 70, ('Lechuga (L)', 1): 24, ('Lechuga (L)', 2): 60, ('Lechuga (L)', 3): 96, ('Lechuga (L)', 4): 120, ('Cebolla (C)', 1): 6, ('Cebolla (C)', 2): 15, ('Cebolla (C)', 3): 24, ('Cebolla (C)', 4): 30, ('Zanahoria (ZN)', 1): 10, ('Zanahoria (ZN)', 2): 25, ('Zanahoria (ZN)', 3): 40, ('Zanahoria (ZN)', 4): 50, ('Pepino (PE)', 1): 18, ('Pepino (PE)', 2): 45, ('Pepino (PE)', 3): 72, ('Pepino (PE)', 4): 90}


# Variable de decisión binaria

\begin{equation}
x_{pzha\ell}=
\begin{cases}
1, & \text{si una caja del producto }p\text{ se coloca en la posición }(z,h,a,\ell),\\
0, & \text{en otro caso},
\end{cases}
\end{equation}



# Función objetivo

$\min \sum_{p \in P} \sum_{z \in Z} \sum_{h \in H_z} \sum_{a \in A_z} \sum_{\ell \in L_z} d_{pz} x_{pzha\ell}  \tag{1}$

# Restricciones

\begin{alignat*}{3}
&\text{s.a.} \quad && \sum_{z \in Z} \sum_{h \in H_z} \sum_{a \in A_z} \sum_{\ell \in L_z} x_{pzha\ell} = n_p, \quad && \forall p \in P, \tag{2} \\[1ex]
&&& \sum_{p \in P} x_{pzha\ell} \le 1, \quad && \forall z \in Z, \, h \in H_z, \, a \in A_z, \, \ell \in L_z, \tag{3} \\[1ex]
&&& \sum_{p \in P} x_{pzha\ell} \le \sum_{p \in P} x_{pzha,\ell-1}, \quad && \forall z \in Z, \, h \in H_z, \, a \in A_z, \, \ell \in L_z : \ell > 1, \tag{4} \\[1ex]
&&& \sum_{p \in P} \sum_{l > \ell}^{|L_z|} w_p x_{pzhal} \le \sum_{p \in P} s_p x_{pzha\ell}, \quad && \forall z \in Z, \, h \in H_z, \, a \in A_z, \, \ell \in L_z : \ell < |L_z|, \tag{5} \\[1ex]
&&& x_{pzha\ell} \in \{0, 1\}, \quad && \forall p \in P, \, z \in Z, \, h \in H_z, \, a \in A_z, \, \ell \in L_z. \tag{6}
\end{alignat*}

In [ ]:

m = gp.Model("bin packing problem")
#Definimos primero el modelo y llamamos gp.Model
# y almacena variables de decisión, la función objetivo y las restricciones

# ============================================================
# VARIABLES DE DECISION
# x[p,z,h,a,l]
# ============================================================

x = m.addVars(P,Z,H,A,L,
              vtype=GRB.BINARY, #SON TIPO BINARIAS
              name="x")

# ============================================================
# FUNCIÓN OBJETIVO
# ============================================================

m.setObjective(

    gp.quicksum(
        d[p,z]*x[p,z,h,a,l] #DETERIORO POR VARIABLE DE DECISION
        for p in P
        for z in Z
        for h in H
        for a in A
        for l in L
    ),

    GRB.MINIMIZE  ### SE REQUIERE EL MINIMO

)

# ============================================================
# RESTRICCIÓN (2)
# Número de cajas de cada producto
# ============================================================

##### la función .addConstr() agrega la restricción al modelo
### En este caso, el modelo se define como m

for p in P:

    m.addConstr(

        gp.quicksum(
            x[p,z,h,a,l]
            for z in Z
            for h in H
            for a in A
            for l in L
        ) == n[p],

        name=f"Demanda_{p}" ###NOMBRE DE LA EC.

    )

# ============================================================
# RESTRICCIÓN (3)
# Una caja por posición (A LO MAS)
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in L:

                m.addConstr(

                    gp.quicksum( ##SUMATORIA
                        x[p,z,h,a,l]
                        for p in P
                    ) <= 1,

                    name=f"Ocupacion_{z}_{h}_{a}_{l}"

                )

# ============================================================
# RESTRICCIÓN (4)
# No puede haber cajas flotando
# ============================================================

for z in Z:
    for h in H:
        for a in A:
            for l in L[:-1]:

                m.addConstr(

                    gp.quicksum(
                        x[p,z,h,a,l]
                        for p in P
                    )

                    <=

                    gp.quicksum(
                        x[p,z,h,a,l-1]
                        for p in P
                    ),

                    name=f"Posicion_{z}_{a}_{l}"

                )


# ============================================================
# RESTRICCIÓN (5)
# Peso máximo soportado
# ============================================================

for z in Z:
    for h in H:
        for a in A:

            # No tiene sentido evaluar el último nivel
            for l in L[:-1]:

                peso_superior = gp.quicksum(

                    w[p] * x[p,z,h,a,ll]

                    for p in P
                    for ll in L
                    if ll > l

                )

                resistencia = gp.quicksum(

                    s[p] * x[p,z,h,a,l]

                    for p in P

                )

                m.addConstr(

                    peso_superior <= resistencia,

                    name=f"Apilamiento_{z}_{h}_{a}_{l}"

                )

# ============================================================
# OPTIMIZAR
# ============================================================

m.optimize()
fig = go.Figure()
# ============================================================
# RESULTADOS
# ============================================================

if m.status == GRB.OPTIMAL:

  ###gurobi se asegura de que el modelo haya tenido éxito

    print("\n")
    print("="*60)
    print("VALOR ÓPTIMO")
    print("="*60)
    print(m.objVal)

    for z in Z:

        print("\n")
        print("="*40)
        print(f"ZONA {z}")
        print("="*40)

        # Hilera dentro de la zona
        for h in H:

            print(f"\n----- Hilera {h} -----")

            # Posiciones dentro de la hilera
            for a in A:

                print(f"\nPosición {a}")

                # Niveles de arriba hacia abajo
                for l in reversed(L):

                    producto = "-"

                    for p in P:

                        if x[p,z,h,a,l].X > 0.5:
                            producto = p
                            break

                    print(f"Nivel {l}: {producto}")
else:

    print("No se encontró solución óptima.")



Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 116 rows, 384 columns and 2016 nonzeros (Min)
Model fingerprint: 0x19d720e9
Model has 384 linear objective coefficients
Variable types: 0 continuous, 384 integer (384 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+01]
  Objective range  [4e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+01]

Found heuristic solution: objective 1970.0000000
Presolve time: 0.01s
Presolved: 116 rows, 384 columns, 2016 nonzeros
Variable types: 0 continuous, 384 integer (384 binary)

Root relaxation: objective 1.420250e+03, 193 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node

# Visualización del contenedor en 3D

# Visualización interactiva del contenedor
**Nota:** En GitHub no aparece, ingrese directamente al código de Google Colab para verlo.

In [ ]:
#### INTENTO DE HACERLO MOVIBLE

In [ ]:


def agregar_caja(fig, x, y, z, dx, dy, dz, color, texto, hover):

    vertices = np.array([
        [x, y, z],
        [x+dx, y, z],
        [x+dx, y+dy, z],
        [x, y+dy, z],
        [x, y, z+dz],
        [x+dx, y, z+dz],
        [x+dx, y+dy, z+dz],
        [x, y+dy, z+dz]
    ])

    I=[0,0,0,1,1,2,4,4,5,6,2,3]
    J=[1,2,3,2,5,3,5,7,6,7,6,7]
    K=[2,3,1,5,6,7,7,6,4,4,1,0]

    fig.add_trace(

        go.Mesh3d(

            x=vertices[:,0],
            y=vertices[:,1],
            z=vertices[:,2],

            i=I,
            j=J,
            k=K,

            color=color,
            opacity=0.95,
            flatshading=True,

            hovertext=hover,
            hoverinfo="text",
            showscale=False

        )

    )

    fig.add_trace(

        go.Scatter3d(

            x=[x+dx/2],
            y=[y+dy/2],
            z=[z+dz/2],

            mode="text",

            text=[texto],

            textfont=dict(
                color="white",
                size=12
            ),

            showlegend=False,

            hoverinfo="skip"

        )

    )

In [ ]:
color_producto = {
    "Fresa (F)": "red",
    "Papa (P)": "saddlebrown",
    "Jitomate (J)": "lightcoral",
    "Manzana (M)": "gold",
    "Lechuga (L)": "lime",
    "Cebolla (C)": "purple",
    "Zanahoria (ZN)": "orange",
    "Pepino (PE)": "darkgreen"
}

In [ ]:
espacio_zona = len(A)+1

for p in P:

    for z in Z:

        for h in H:

            for a in A:

                for l in L:

                    if x[p,z,h,a,l].X > 0.5:

                        X = (z-1)*espacio_zona
                        Y = a-1
                        Zplot = l-1

                        hover = (
                            f"<b>Producto:</b> {p}<br>"
                            f"<b>Zona:</b> {z}<br>"
                            f"<b>Hilera:</b> {h}<br>"
                            f"<b>Posición:</b> {a}<br>"
                            f"<b>Nivel:</b> {l}<br>"
                            f"<b>Peso:</b> {w[p]} kg<br>"
                            f"<b>Resistencia:</b> {s[p]} kg<br>"
                            f"<b>Temperatura:</b> {theta[z]} °C"
                        )

                        agregar_caja(

                            fig,

                            X,
                            Y,
                            Zplot,

                            0.9,
                            0.9,
                            0.9,

                            color_producto[p],

                            p,

                            hover

                        )

In [ ]:
for z in Z:

    X = (z-1)*espacio_zona

    vertices = np.array([
        [X,0,0],
        [X+1,0,0],
        [X+1,len(A),0],
        [X,len(A),0],
        [X,0,len(L)],
        [X+1,0,len(L)],
        [X+1,len(A),len(L)],
        [X,len(A),len(L)]
    ])

    fig.add_trace(

        go.Mesh3d(

            x=vertices[:,0],
            y=vertices[:,1],
            z=vertices[:,2],

            i=[0,0,0,1,1,2,4,4,5,6,2,3],
            j=[1,2,3,2,5,3,5,7,6,7,6,7],
            k=[2,3,1,5,6,7,7,6,4,4,1,0],

            color="gray",

            opacity=0.08,

            hoverinfo="skip",

            showscale=False

        )

    )

In [ ]:
fig.update_layout(

    title="Distribución óptima de cajas",

    scene=dict(

        xaxis=dict(
            title="Zona",
            tickvals=[(z-1)*espacio_zona for z in Z],
            ticktext=[f"Z{z}" for z in Z]
        ),

        yaxis=dict(
            title="Posición",
            tickvals=[a-1 for a in A],
            ticktext=[f"A{a}" for a in A]
        ),

        zaxis=dict(
            title="Nivel",
            tickvals=[l-1 for l in L],
            ticktext=[f"N{l}" for l in L]
        ),

        aspectmode="data"

    ),

    width=1200,

    height=750

)

fig.show()